## Instruction Finetuning
<div class="alert alert-info" role="alert">
Instruction fine-tuning is a process of training a pre-trained AI model, especially a large language model (LLM), on a dataset of instructions and their corresponding desired outputs. This helps the model better understand and follow human instructions to perform specific tasks, improving its ability to generalize and respond to novel prompts across a wide range of domains.
</div>

In [ ]:
import json
import os
import urllib
import ssl
import torch

def download_file(url, file_path):
    ssl_context = ssl.create_default_context() # Create an SSL context that does not verify certificates
    ssl_context.check_hostname = False # Disable hostname checking
    ssl_context.verify_mode = ssl.CERT_NONE # Disable certificate verification

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url, context = ssl_context) as response:
            text_data = response.read().decode('utf-8')
        with open(file_path, 'w', encoding = 'utf-8') as f:
            f.write(text_data)
    else:
        with open(file_path, 'r', encoding= 'utf-8') as f:
            text_data = f.read()
    with open(file_path, 'r', encoding= 'utf-8') as f:
        data = json.load(f)
    return data

file_path = 'alpaca_data.json'
data_url = 'https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json'
data = download_file(data_url, file_path)
data = data[:25000]
data[:5]



[{'instruction': 'Give three tips for staying healthy.',
  'input': '',
  'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'},
 {'instruction': 'What are the three primary colors?',
  'input': '',
  'output': 'The three primary colors are red, blue, and yellow.'},
 {'instruction': 'Describe the structure of an atom.',
  'input': '',
  'output': 'An atom is made up of a nucleus, which contains protons and neutrons, surrounded by electrons that travel in orbits around the nucleus. The protons and neutrons have a positive charge, while the electrons have a negative charge, resulting in an overall neutral atom. The number of each particle determines the atomic number and the type of atom.'},
 {'instruction': 'How can we reduce air pollution?',
  'input': '',
  'output': 'There are a number of ways to reduce air pollution, such

In [ ]:
len(data)

25000

In [ ]:
data[50]

{'instruction': 'Edit the following sentence to make it more concise.',
 'input': 'He ran to the bus stop in order to catch the bus that was due to arrive in five minutes.',
 'output': 'He ran to the bus stop, due to arrive in five minutes.'}

We are using the alpaca dataset for instruction finetuning. The dataset contains 52,000 instruction-following examples generated from OpenAI's text-davinci-003 model.

#### Converting the dataset into alpaca format

In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n {entry["instruction"]}"
    )

    input_text = f"\n\n### Input:\n {entry["input"]}" if entry["input"] else ""

    return instruction_text + input_text

model_input = format_input(data[112])
desired_output = f"\n\n### Response:\n {data[10]["output"]}"

print(model_input + desired_output)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
 Identify the incorrect word and suggest a better version.

### Input:
 The waitress served a humonguous burger.

### Response:
 Julius Caesar was assassinated by a group of up to 60 conspirators, led by Gaius Cassius Longinus and Marcus Junius Brutus, in the Senate House on the Ides of March (15 March) of 44 BC.


In [ ]:
## partioning the dataset into train and test sets
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
val_portion = len(data) - train_portion - test_portion

training_data = data[:train_portion]
testing_data = data[train_portion:train_portion + test_portion]
validation_data = data[train_portion + test_portion:]

## Converting the data into batch format for training

1. Format the data using prompt template
2. Tokenize the formated data
3. Adjust the same lenght with padding tokens
4. Create target token IDs for training
5. replace padding tokens with placeholder -100(used in pytorch to ignore certain tokens in loss computation

)

In [ ]:
## we will be using pytorch Dataset to curate the data for training

from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        self.encoded_text = []

        for entry in data:
            instruction_and_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_and_input + response_text

            self.encoded_text.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_text[index]

    def __len__(self):
        return len(self.data)


In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
def custom_collate_fn(
        batch,
        pad_token_id = 50256,
        ignore_index = -100,
        allowed_max_len = 1024,
        device = 'cpu'
):
    batch_max_len = max(len(item)+1 for item in batch)## added by 1 for the eos token

    input_lst, target_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id] ## adding eos token at the end
        padded = (
            new_item + [pad_token_id] * (batch_max_len - len(new_item)) # padding to max length
        )

        inputs = torch.tensor(padded[:-1]) ## removing the extra padded token
        targets = torch.tensor(padded[1:]) ## shifting by 1 for target

        # replace all but the first padding tokens in targets by ignore_index
        mask= targets == pad_token_id ## create a mask for padding tokens
        indices = torch.nonzero(mask).squeeze() ## get the indices of padding tokens
        if indices.numel()> 1:
            targets[indices[1:]] = ignore_index ## replace all but the first padding token with ignore_index
         # New: Optionally truncate to maximum sequence length
        if allowed_max_len is not None:
            inputs = inputs[:allowed_max_len]
            targets = targets[:allowed_max_len]
        input_lst.append(inputs)
        target_lst.append(targets)
    ## convert list of inputs to tensor and transfer to target device
    input_tensor = torch.stack(input_lst).to(device)
    target_tensor = torch.stack(target_lst).to(device)
    return input_tensor, target_tensor



In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
## example usage of the dataset and collate function
import torch
input_1 = [0, 1, 2, 3]
input_2 = [4, 5, 6]
input_3 = [7, 8, 9, 10, 11]
batch = [input_1, input_2, input_3]
padded_batch = custom_collate_fn(batch, pad_token_id=50256, allowed_max_len=1024, device='cuda')
print(padded_batch)

(tensor([[    0,     1,     2,     3, 50256],
        [    4,     5,     6, 50256, 50256],
        [    7,     8,     9,    10,    11]], device='cuda:0'), tensor([[    1,     2,     3, 50256,  -100],
        [    5,     6, 50256,  -100,  -100],
        [    8,     9,    10,    11, 50256]], device='cuda:0'))


## Creating dataloader

We moved the data onto the target device (for example, the GPU memory when device="cuda") in the main training loop. Having this as part of the collate function offers the advantage of performing this device transfer process as a background process outside the training loop, preventing it from blocking the GPU during model training.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Next, to reuse the chosen device setting in custom_collate_fn when we plug it into the PyTorch DataLoader class later in this section, we use the partial function from Python's functools standard library to create a new version of the function with the device argument pre-filled.

Additionally, we set the allowed_max_length to 1024, which truncates the data to the maximum context length supported by the GPT-2 model we finetune later in this chapter:

In [ ]:
from functools import partial
customized_collate_fn = partial(custom_collate_fn, device=device, allowed_max_len=1024)

In [ ]:
import torch
from torch.utils.data import DataLoader
num_workers = 0
batch_size = 1  # Reduced from 4 to 2 for memory savings

torch.manual_seed(42)

train_dataset = InstructionDataset(
    training_data, tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    collate_fn = customized_collate_fn,
    shuffle = True,
    drop_last = True,
    num_workers = num_workers
)

val_dataset = InstructionDataset(
    validation_data, tokenizer
)
val_loader = DataLoader(
    val_dataset,
    batch_size = batch_size,
    collate_fn = customized_collate_fn,
    shuffle = False,
    drop_last = False,
    num_workers = num_workers
)

test_dataset = InstructionDataset(
    testing_data, tokenizer
)
test_loader = DataLoader(
    test_dataset,
    batch_size = batch_size,
    collate_fn = customized_collate_fn,
    shuffle = False,
    drop_last = False,
    num_workers = num_workers
)

In [ ]:
print("Train_loder")
count = 0
for inputs, targets, in train_loader:
    count += 1
    print("Inputs:", inputs.shape)
    print("Targets:", targets.shape )
print(count)

Streaming output truncated to the last 5000 lines.
Targets: torch.Size([1, 37])
Inputs: torch.Size([1, 191])
Targets: torch.Size([1, 191])
Inputs: torch.Size([1, 168])
Targets: torch.Size([1, 168])
Inputs: torch.Size([1, 66])
Targets: torch.Size([1, 66])
Inputs: torch.Size([1, 131])
Targets: torch.Size([1, 131])
Inputs: torch.Size([1, 146])
Targets: torch.Size([1, 146])
Inputs: torch.Size([1, 132])
Targets: torch.Size([1, 132])
Inputs: torch.Size([1, 67])
Targets: torch.Size([1, 67])
Inputs: torch.Size([1, 96])
Targets: torch.Size([1, 96])
Inputs: torch.Size([1, 50])
Targets: torch.Size([1, 50])
Inputs: torch.Size([1, 61])
Targets: torch.Size([1, 61])
Inputs: torch.Size([1, 68])
Targets: torch.Size([1, 68])
Inputs: torch.Size([1, 78])
Targets: torch.Size([1, 78])
Inputs: torch.Size([1, 68])
Targets: torch.Size([1, 68])
Inputs: torch.Size([1, 54])
Targets: torch.Size([1, 54])
Inputs: torch.Size([1, 98])
Targets: torch.Size([1, 98])
Inputs: torch.Size([1, 100])
Targets: torch.Size([1, 10

## Loading the GPT 2 model

In [1]:

import os
import requests  # Make sure requests is installed
import json
import numpy as np
import tensorflow as tf
from tqdm import tqdm

def download_and_load_gpt2(model_size, models_dir):
    # Validate model size
    allowed_sizes = ("124M", "355M", "774M", "1558M")
    if model_size not in allowed_sizes:
        raise ValueError(f"Model size not in {allowed_sizes}")

    # Define paths
    model_dir = os.path.join(models_dir, model_size)
    base_url = "https://openaipublic.blob.core.windows.net/gpt-2/models"
    filenames = [
        "checkpoint", "encoder.json", "hparams.json",
        "model.ckpt.data-00000-of-00001", "model.ckpt.index",
        "model.ckpt.meta", "vocab.bpe"
    ]

    # Download files
    os.makedirs(model_dir, exist_ok=True)
    for filename in filenames:
        file_url = os.path.join(base_url, model_size, filename)
        file_path = os.path.join(model_dir, filename)
        download_file(file_url, file_path)

    ## We have reached here until now ---> we have downloaded the files on our local machine.

    # Load settings and params
    tf_ckpt_path = tf.train.latest_checkpoint(model_dir)
    settings = json.load(open(os.path.join(model_dir, "hparams.json")))
    params = load_gpt2_params_from_tf_ckpt(tf_ckpt_path, settings)

    return settings, params

def download_file(url, destination):
    try:
        # Send a GET request to download the file, disabling SSL verification
        response = requests.get(url, stream=True, verify=False)

        # Get the total file size from headers, defaulting to 0 if not present
        file_size = int(response.headers.get("content-length", 0))

        # Check if file exists and has the same size
        if os.path.exists(destination):
            file_size_local = os.path.getsize(destination)
            if file_size == file_size_local:
                print(f"File already exists and is up-to-date: {destination}")
                return

        # Define the block size for reading the file
        block_size = 1024  # 1 Kilobyte

        # Initialize the progress bar with total file size
        progress_bar_description = url.split("/")[-1]  # Extract filename from URL
        with tqdm(total=file_size, unit="iB", unit_scale=True, desc=progress_bar_description) as progress_bar:
            # Open the destination file in binary write mode
            with open(destination, "wb") as file:
                # Iterate over the file data in chunks
                for chunk in response.iter_content(block_size):
                    progress_bar.update(len(chunk))  # Update progress bar
                    file.write(chunk)  # Write the chunk to the file

    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
        print(f"Please check the URL: {url}")

def load_gpt2_params_from_tf_ckpt(ckpt_path, settings):
    # Initialize parameters dictionary with empty blocks for each layer
    params = {"blocks": [{} for _ in range(settings["n_layer"])]}

    # Iterate over each variable in the checkpoint
    for name, _ in tf.train.list_variables(ckpt_path):
        # Load the variable and remove singleton dimensions
        variable_array = np.squeeze(tf.train.load_variable(ckpt_path, name))

        # Process the variable name to extract relevant parts
        variable_name_parts = name.split("/")[1:]  # Skip the 'model/' prefix

        # Identify the target dictionary for the variable
        target_dict = params
        if variable_name_parts[0].startswith("h"):
            layer_number = int(variable_name_parts[0][1:])
            target_dict = params["blocks"][layer_number]

        # Recursively access or create nested dictionaries
        for key in variable_name_parts[1:-1]:
            target_dict = target_dict.setdefault(key, {})

        # Assign the variable array to the last key
        last_key = variable_name_parts[-1]
        target_dict[last_key] = variable_array

    return params

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

NameError: name 'torch' is not defined

In [3]:
from torch import nn

class GELU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5 * x * (1 + torch.tanh(
        torch.sqrt(torch.tensor(2.0/torch.pi)) * (x + 0.044715 * torch.pow(x,3)) ## we use approximation instead of CDF and it is used in GPT 2 model
    ))

class MutliHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias = False):
    super().__init__()
    assert (d_out % num_heads) == 0, \
      "d_out must be divisible by the num_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out // num_heads ## reducing the projection dim to match the desired outpit dim
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.out_proj = nn.Linear(d_out, d_out) ## lear layer to combine the head output
    self.dropout = nn.Dropout(dropout)
    self.register_buffer( ## register_buffer method is used to manage and save tensors that are part of a module's state but are not trainable parameters.When you move your model to a different device (e.g., from CPU to GPU) using model.to(device), any tensors registered as buffers will automatically be moved to that same device
        "mask",
        torch.triu(torch.ones(context_length, context_length), diagonal=1)
    )
  def forward(self, x): ## lets x = "the cat sleeps" and num of head is 2
    b,num_tokens, d_in = x.shape
    query = self.W_query(x)
    key = self.W_key(x)
    value = self.W_value(x)

    ## we implicitly split the matrix by adding a 'num_heads' dimension
    keys = key.view(b, num_tokens, self.num_heads, self.head_dim)
    values = value.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = query.view(b, num_tokens, self.num_heads, self.head_dim)

    ## currently the shape is [1,3,2,3] if the batch size is 1, num of tokens is 3, heads is 2 and head dim is 3
    ## so lets transpose it for index 1,2 to make it [1,2,3,3]
    keys = keys.transpose(1,2)
    values = values.transpose(1,2)
    queries = queries.transpose(1,2)

    ## computing the scaled attention score with causal mask
    attn_score = queries @ keys.transpose(2,3)

    ## original mask truncated to number of tokens and converted to boolean
    mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
    attn_score.masked_fill_(mask_bool, -torch.inf)
    attn_weights = torch.softmax(attn_score/self.head_dim**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    ## shape: (b, num_tokens, num_heads, head_dim)
    context_vecs = (attn_weights @ values).transpose(1,2)
    context_vecs = context_vecs.contiguous().view(b,num_tokens, self.d_out)
    context_vecs= self.out_proj(context_vecs)
    return context_vecs

class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
        GELU(),
        nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"] )
    )
  def forward(self,x):
    return self.layers(x)

class TransformerBlock(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.att = MutliHeadAttention(
        d_in = cfg["emb_dim"],
        d_out= cfg["emb_dim"],
        context_length= cfg["context_length"],
        dropout= cfg["drop_rate"],
        num_heads= cfg["n_heads"],
        qkv_bias = cfg["qkv_bias"]
    )
    self.ff = FeedForward(cfg)
    self.norm1 = LayerNorm(cfg["emb_dim"])
    self.norm2 = LayerNorm(cfg["emb_dim"])
    self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

  def forward(self, x):
    ## adding shortcut for attention block
    shortcut = x
    x = self.norm1(x)
    x = self.att(x)
    x = self.drop_shortcut(x)
    x = shortcut + x

    ### adding shortcut connection for feedforward block
    shortcut = x
    x = self.norm2(x)
    x = self.ff(x)
    x = self.drop_shortcut(x)
    x = x + shortcut

    return x

## coding Layer normalization class

class LayerNorm(nn.Module):
  def __init__(self, embed_dim):
    super().__init__()
    self.eps = 1e-5
    self.scale = nn.Parameter(torch.ones(embed_dim)) ## parameter of the same dim of input that LLM automatically adjust during training, this allows model to learn appropriate scaling and shifting that best suits the data it process
    self.shift = nn.Parameter(torch.zeros(embed_dim))

  def forward(self,x):
    mean = x.mean(dim = -1, keepdim = True)
    var = x.var(dim= -1, keepdim= True, unbiased = False) ## unbiased false turns off the bessels correction
    norm_x = (x-mean)/ torch.sqrt(var+self.eps) # added eps to ignore division by 0
    return self.scale*norm_x + self.shift


class GPT(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.tok_emb = nn.Embedding(cfg["vocab_size"],cfg["emb_dim"]) ## number of embedding, and token dimension(768)
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"]) ## it will take only the context length and dimenstion
    self.drop_emb = nn.Dropout(cfg["drop_rate"])
    self.trf_blocks = nn.Sequential(
        *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )
    self.final_norm = LayerNorm(cfg["emb_dim"])
    self.out_head = nn.Linear(
        cfg["emb_dim"], cfg["vocab_size"], bias=False
    )

  def forward(self,in_idx):
    batch_size, seq_len = in_idx.shape
    ## inputing tokens
    token_embed = self.tok_emb(in_idx)
    ## positional encoding
    pos_embeds = self.pos_emb(torch.arange(seq_len, device = in_idx.device))
    ## adding encodings
    x = token_embed + pos_embeds
    ## adding dropour layer
    x = self.drop_emb(x)
    ## now going through all transformer blocks => 12
    x = self.trf_blocks(x)
    ## final normalization layer
    x = self.final_norm(x)
    ## output
    logits = self.out_head(x) ## shape : [batch, num_tokens, vocab_size ]
    return logits

In [4]:
CONFIG= {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True

}

In [5]:
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12 },
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16 },
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20 },
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25 },
}

In [6]:
model_name = "gpt2-medium (355M)"


In [7]:
GPT_CONFIG_355M = CONFIG.copy()
GPT_CONFIG_355M.update(model_configs[model_name])
GPT_CONFIG_355M

{'vocab_size': 50257,
 'context_length': 1024,
 'drop_rate': 0.0,
 'qkv_bias': True,
 'emb_dim': 1024,
 'n_layers': 24,
 'n_heads': 16}

In [ ]:
settings, params  = download_and_load_gpt2(
    model_size = "355M",
    models_dir = "gpt2"
)

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 269kiB/s]
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
encoder.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 3.52MiB/s]
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verificat

In [ ]:
model = GPT(GPT_CONFIG_355M).to(device)

In [ ]:
model

GPT(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MutliHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MutliHeadAttention(
        (W_query): Linear(in_fea

In [ ]:
import numpy as np
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))
def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])

    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])



In [ ]:
load_weights_into_gpt(model,params)
model = model.to(device)
model.eval()

GPT(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MutliHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MutliHeadAttention(
        (W_query): Linear(in_fea

In [ ]:
torch.manual_seed(123)

input_text = format_input(data[0])
print(input_text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
 Give three tips for staying healthy.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
def text_to_token_ids(text, tokenizer, device= device):
    encoded = tokenizer.encode(text, allowed_special = {'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0).to(device)  # Move to device
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)  # remove the batch
    return tokenizer.decode(flat.tolist())

In [ ]:
## so applying both strategies
## logits -> top-k sampling -> logits/temp -> softmax -> sample from multinomial

## final generation code with temperation and top-k
def generate(model, idx, max_new_tokens, context_size, temperature = 0.0, top_k = None, eos_id = None):

  for _ in range(max_new_tokens):
    # Ensure slicing does not exceed the current sequence length
    idx_cond = idx[:, max(0, idx.shape[1] - context_size):]
    with torch.no_grad():
      logits = model(idx_cond)
    logits = logits[:,-1,:]

    if top_k is not None:
      top_logits, top_pos = torch.topk(logits, top_k)
      min_value = top_logits[:,-1]
      logits = torch.where(
          logits< min_value, torch.tensor(float("-inf")).to(logits.device), logits
      )

    if temperature > 0.0:
      logits = logits/temperature

      probs = torch.softmax(logits, dim = -1)
      idx_next = torch.multinomial(probs, num_samples = 1)
    else:
      idx_next = torch.argmax(logits, dim = -1, keepdim = True)
    if eos_id is not None and idx_next == eos_id:
      break
    idx = torch.cat((idx, idx_next),dim = 1)
  return idx

In [ ]:
token_ids = generate(
    model,
    text_to_token_ids(input_text,tokenizer, device = device),
    max_new_tokens=34,
    context_size=1024,
    eos_id = 50256
)
generated_text = token_ids_to_text(token_ids,tokenizer)
print(generated_text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
 Give three tips for staying healthy.

### Response:

### Instruction:

### Response:

### Instruction:

### Response:

### Instruction:

### Response


In [ ]:
response_text = generated_text[len(input_text):].strip()
print(response_text) ## This code snippet removes the input text from the beginning of the generated_text

### Response:

### Instruction:

### Response:

### Instruction:

### Response:

### Instruction:

### Response


    
As we can see from the output, the pretrained model is not yet capable of correctly
following the given instruction.

While it does create a "Response" section, it simply repeats
the original input sentence .

In the upcoming section, we implement the finetuning process to improve the model's
ability to comprehend and appropriately respond to such requests.


## Fintuning the LLM on Instructuion dataset



In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

0

## Memory Optimization Strategies

**Applied optimizations:**
1. **Reduced batch size**: 2 instead of 8
2. **Gradient accumulation**: Accumulate 2 steps = effective batch size of 4
3. **Explicit memory cleanup**: Delete intermediate tensors and call `torch.cuda.empty_cache()`
4. **Periodic cache clearing**: Every 10 steps during training
5. **Using GPT-2 Medium (355M)**: If still having issues, switch to GPT-2 Small (124M)

**If memory issues persist**, try:
- Switch to GPT-2 Small: Change `model_size = "124M"` and update config
- Enable gradient checkpointing (trades compute for memory)
- Reduce sequence length in `allowed_max_len`

In [ ]:
model

GPT(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MutliHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MutliHeadAttention(
        (W_query): Linear(in_fea

In [ ]:
## Defining the calculate loss batch function

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)

    ## getting logits
    logits = model(input_batch)

    # Flatten and calculate loss
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten()
    )

    # CRITICAL: Delete intermediate tensors to free memory
    del logits
    torch.cuda.empty_cache()

    return loss

def calc_loss_loader(data_loader, model, device, num_batchs = None):
    total_loss = 0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batchs is None:
        num_batchs = len(data_loader)

    else:
        num_batchs = min(num_batchs, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batchs:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batchs


Gradient Accumulation Explained
Gradient accumulation is a technique to simulate larger batch sizes when you don't have enough GPU memory. Instead of updating weights after every batch, you accumulate gradients over multiple small batches before updating.


In [ ]:
## first check all the input bathes and target batches in train and validation are according to tht contenxt lenght ?
for inputs, targets in train_loader:
    print("Input batch shape:", inputs.shape)
    print("Target batch shape:", targets.shape)
    assert inputs.shape[1] <= GPT_CONFIG_355M["context_length"], "Input batch exceeds context length"
    assert targets.shape[1] <= GPT_CONFIG_355M["context_length"], "Target batch exceeds context length"
    # break  # Just check the first batch

Streaming output truncated to the last 5000 lines.
Input batch shape: torch.Size([1, 77])
Target batch shape: torch.Size([1, 77])
Input batch shape: torch.Size([1, 89])
Target batch shape: torch.Size([1, 89])
Input batch shape: torch.Size([1, 110])
Target batch shape: torch.Size([1, 110])
Input batch shape: torch.Size([1, 66])
Target batch shape: torch.Size([1, 66])
Input batch shape: torch.Size([1, 51])
Target batch shape: torch.Size([1, 51])
Input batch shape: torch.Size([1, 144])
Target batch shape: torch.Size([1, 144])
Input batch shape: torch.Size([1, 83])
Target batch shape: torch.Size([1, 83])
Input batch shape: torch.Size([1, 109])
Target batch shape: torch.Size([1, 109])
Input batch shape: torch.Size([1, 68])
Target batch shape: torch.Size([1, 68])
Input batch shape: torch.Size([1, 58])
Target batch shape: torch.Size([1, 58])
Input batch shape: torch.Size([1, 57])
Target batch shape: torch.Size([1, 57])
Input batch shape: torch.Size([1, 150])
Target batch shape: torch.Size([1,

In [ ]:
## creating finetuning loop with gradient accumulation
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, eval_iter)
    model.train()
    return train_loss, val_loss

def finetune_model(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):

    # list to track the losses
    train_losses, val_losses, track_tokens_seen = [], [], []

    token_seen, global_step = 0, -1

    # Gradient accumulation settings
    accumulation_steps = 4  # Effective batch size = 4 * 2 = 8

    # loop
    for epoch in range(num_epochs):
        model.train()

        for batch_idx, (input_batch, target_batch) in enumerate(train_loader):
            # Calculate loss
            loss = calc_loss_batch(input_batch, target_batch, model, device)

            # Scale loss by accumulation steps
            loss = loss / accumulation_steps

            # Backward pass
            loss.backward()

            # Only update weights every accumulation_steps
            if (batch_idx + 1) % accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()

                # Clear cache periodically
                if global_step % 10 == 0:
                    torch.cuda.empty_cache()

            token_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(token_seen)
                print(
                    f"Ep {epoch+1} | Step {global_step} | Tokens {token_seen:,} | "
                    f"Train {train_loss:.4f} | Val {val_loss:.4f}"
                )

        # Make sure to step optimizer at end of epoch if there are remaining gradients
        if (batch_idx + 1) % accumulation_steps != 0:
            optimizer.step()
            optimizer.zero_grad()

    return train_losses, val_losses, track_tokens_seen

In [ ]:
def calculate_training_steps(dataset_size, batch_size, accumulation_steps, num_epochs=1, drop_last=True):
    """
    Calculate the number of training steps for your configuration.

    Args:
        dataset_size: Total number of samples in training set
        batch_size: Number of samples per batch
        accumulation_steps: Number of gradient accumulation steps
        num_epochs: Number of training epochs
        drop_last: Whether to drop the last incomplete batch

    Returns:
        Dictionary with training metrics
    """
    # Calculate batches per epoch
    if drop_last:
        batches_per_epoch = dataset_size // batch_size
    else:
        batches_per_epoch = (dataset_size + batch_size - 1) // batch_size

    # Calculate optimizer steps per epoch
    optimizer_steps_per_epoch = batches_per_epoch // accumulation_steps

    # Calculate total for all epochs
    total_batches = batches_per_epoch * num_epochs
    total_optimizer_steps = optimizer_steps_per_epoch * num_epochs

    # Calculate effective batch size
    effective_batch_size = batch_size * accumulation_steps

    return {
        "dataset_size": dataset_size,
        "batch_size": batch_size,
        "accumulation_steps": accumulation_steps,
        "effective_batch_size": effective_batch_size,
        "batches_per_epoch": batches_per_epoch,
        "optimizer_steps_per_epoch": optimizer_steps_per_epoch,
        "total_epochs": num_epochs,
        "total_batches": total_batches,
        "total_optimizer_steps": total_optimizer_steps,
        "samples_dropped": dataset_size - (batches_per_epoch * batch_size) if drop_last else 0
    }

# Calculate for your configuration
stats = calculate_training_steps(
    dataset_size=train_dataset.__len__(),  # Your training set size
    batch_size=1,        # Current batch size
    accumulation_steps = 4, # Current accumulation steps
    num_epochs=1,
    drop_last=True
)

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)
for key, value in stats.items():
    print(f"{key:.<35} {value:,}")
print("=" * 60)

TRAINING CONFIGURATION
dataset_size....................... 21,250
batch_size......................... 1
accumulation_steps................. 4
effective_batch_size............... 4
batches_per_epoch.................. 21,250
optimizer_steps_per_epoch.......... 5,312
total_epochs....................... 1
total_batches...................... 21,250
total_optimizer_steps.............. 5,312
samples_dropped.................... 0


In [ ]:
## Clear memory before training
import gc
# Only delete if variables exist
if 'train_losses' in locals():
    del train_losses
if 'val_losses' in locals():
    del val_losses
if 'token_seen' in locals():
    del token_seen
torch.cuda.empty_cache()
gc.collect()

# Monitor GPU memory
print(f"GPU allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

## initializing finetuning loop
import time

start = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)
num_epochs = 1

train_losses, val_losses, token_seen = finetune_model(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(validation_data[0]), tokenizer=tokenizer
)

end_time = time.time()
print(f"Total finetuning time for {num_epochs} epoch(s): {(end_time - start)/60:.2f} minutes")

GPU allocated: 1.62 GB
GPU reserved: 1.81 GB
Ep 1 | Step 0 | Tokens 123 | Train 3.9044 | Val 2.9754
Ep 1 | Step 5 | Tokens 586 | Train 2.6080 | Val 2.4031
Ep 1 | Step 10 | Tokens 1,175 | Train 2.4511 | Val 2.1944
Ep 1 | Step 15 | Tokens 1,729 | Train 1.8175 | Val 1.8736
Ep 1 | Step 20 | Tokens 2,122 | Train 1.8878 | Val 1.7451
Ep 1 | Step 25 | Tokens 2,825 | Train 1.7094 | Val 1.6634
Ep 1 | Step 30 | Tokens 3,412 | Train 1.6339 | Val 1.6294
Ep 1 | Step 35 | Tokens 3,815 | Train 1.7302 | Val 1.6145
Ep 1 | Step 40 | Tokens 4,306 | Train 1.5536 | Val 1.6088
Ep 1 | Step 45 | Tokens 4,875 | Train 1.5996 | Val 1.6020
Ep 1 | Step 50 | Tokens 5,366 | Train 1.2782 | Val 1.5960
Ep 1 | Step 55 | Tokens 5,983 | Train 1.6341 | Val 1.5921
Ep 1 | Step 60 | Tokens 6,397 | Train 1.5789 | Val 1.5857
Ep 1 | Step 65 | Tokens 6,895 | Train 1.6233 | Val 1.5808
Ep 1 | Step 70 | Tokens 7,292 | Train 1.8717 | Val 1.5792
Ep 1 | Step 75 | Tokens 8,761 | Train 1.1597 | Val 1.5722
Ep 1 | Step 80 | Tokens 9,218 | T

In [ ]:
# Monitor GPU memory
print(f"GPU allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

GPU allocated: 4.65 GB
GPU reserved: 8.04 GB


In [ ]:

## training and validation loss curves

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_loss_curves(epoch_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    ax1.plot(epoch_seen, train_losses, label='Train Loss', color='blue')
    ax1.plot(epoch_seen, val_losses, label='Validation Loss',linestyle = "-.", color='orange')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend(loc='upper right')
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
    # Create a second x-axis for tokens seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(tokens_seen, train_losses, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig("loss-plot.pdf")
    plt.show()

In [ ]:
## Extracting and saving the response using test set

torch.manual_seed(123)

for i in range(5):
    input_text = format_input(testing_data[i])
    token_ids = generate(
        model,
        text_to_token_ids(input_text,tokenizer, device = device),
        max_new_tokens=34,
        context_size=1024,
        eos_id = 50256
    )
    generated_text = token_ids_to_text(token_ids,tokenizer)
    response_text = generated_text[len(input_text):].strip()
    print(f"Input {i+1}:\n{input_text}\n")
    print(f"Response {i+1}:\n{response_text}\n")


## saving the finetuned model
model_save_path = "finetuned_gpt2_medium.pth"
torch.save(model.state_dict(), model_save_path)


Input 1:
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
 Generate a math equation and solve it.

Response 1:
### Response:
2 x 3 = 5

Input 2:
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
 Remember the appointment your friend has on Friday.

### Input:
 My friend Jane has an appointment on Friday.

Response 2:
### Response:
Jane has an appointment on Friday.

Input 3:
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
 Find the total population of Germany

Response 3:
in 2020.

### Response:
According to the latest estimates, the population of Germany in 2020 is around 1.2 billion people.

Input 4:
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
 Identify how many grams of sugar are in 

In [ ]:
## saving the model in google drive
from google.colab import drive
drive.mount('/content/drive')
model_save_path = "/content/drive/MyDrive/finetuned_gpt2_medium.pth"
torch.save(model.state_dict(), model_save_path)


Mounted at /content/drive


In [10]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GPT(GPT_CONFIG_355M).to(device)

In [12]:
## load the model
import torch
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/finetuned_gpt2_medium.pth"
model.to(device)





Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


GPT(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MutliHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MutliHeadAttention(
        (W_query): Linear(in_fea